# Validation vs Enriched two-way invariance (variant of NB_2_cfa_as_reg)

Searches for **scalar invariance between the validation vs enriched samples only** (single `val_en` pair). Identical machinery and conventions to `NB_2_cfa_as_reg.ipynb`, with **all outputs isolated in `data/cfa_en_val/`** so nothing collides with the 3-way pipeline's `data/cfa/` or the other two-way variants. The V_EN baseline rows are reused from the 3-way run where available (seed-identical tests); the stepwise stages always run fresh. Note: the package's progress prints say "3-way" — with a single pair supplied they mean "all supplied pairs". Run headless with `./notebooks/run_overnight.sh NB_2_cfa_as_reg_en_val.ipynb`.

# Prep

## Import stuff

In [1]:
from pathlib import Path
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
#import seaborn as sns
#from matplotlib import pyplot as plt
#%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import rpy2
#import pingouin as pg
from itertools import combinations
import openpyxl
from contextlib import redirect_stdout
import random
import math
import platform

## Some magical magic to make the R stuff work

In [2]:
import os
os.environ["OMP_NUM_THREADS"], os.environ["OPENBLAS_NUM_THREADS"], os.environ["MKL_NUM_THREADS"] 

('1', '1', '1')

In [3]:
# The rpy2/R session and CFA machinery now live in the hitop_cfa package.
# Importing it starts the embedded R session: it sets the BLAS threading
# env vars (if unset) and loads base, utils, lavaan, and the patched semTools.
import hitop_cfa
from hitop_cfa.r_env import (ro, rbase, utils, lavaan, semtools,
                             RRuntimeError, pandas2ri, localconverter)

import rpy2.ipython.html
rpy2.ipython.html.init_printing()

# Paths

In [4]:
# paths where to save preprocessed data files
log_dir = Path('./log')
log_dir.mkdir(exist_ok=True)
dat_dir = Path('../data/')
val_dir = dat_dir / 'ValSample'
fin_dir = dat_dir / 'finaldata'
cfa_dir = dat_dir / 'cfa_en_val'
cfa_dir.mkdir(exist_ok=True)
path_save_val = fin_dir / 'dat_val.csv'
path_save_dat_gp_grid1st_norecontact = fin_dir / 'dat_gp_grid1st_norecontact.csv'
path_save_dat_en_grid1st_norecontact = fin_dir / 'dat_en_grid1st_norecontact.csv'
path_save_dat_gp_grid1st_full = fin_dir / 'dat_gp_grid1st_full.csv'
path_save_dat_en_grid1st_full = fin_dir / 'dat_en_grid1st_full.csv'
path_save_dat_gp_gridall_full = fin_dir / 'dat_gp_gridall_full.csv'
path_save_dat_en_gridall_full = fin_dir / 'dat_en_gridall_full.csv'
# path_save_dat_gp_gridall_recontact = '../../data/finaldata/dat_gp_gridall_recontact.csv'
# path_save_dat_en_gridall_recontact = '../../data/finaldata/dat_en_gridall_recontact.csv'
# helped file for cfa
helpfile_dir = cfa_dir / 'temp'
helpfile_dir.mkdir(exist_ok=True, parents=True)
path_to_helpfile = helpfile_dir / 'cfa_temp.csv'
path_to_cogmood_questions = dat_dir / 'cogmood_questions.csv'
path_to_item_lookup = val_dir / 'Internalizing-Somatoform Items_DW.xlsx'

In [5]:
import datetime
import traceback

# Overnight robustness: every long loop below wraps its per-scale work in
# try/except and calls this on failure, so ONE bad scale cannot kill the
# whole run. Errors are printed AND appended (with tracebacks) to
# cfa_dir/run_errors.log; run_error_records collects them for the summary
# cell at the end of the notebook.
run_error_records = []


def record_run_error(stage, scale, exc):
    stamp = datetime.datetime.now().isoformat(timespec='seconds')
    msg = f"[{stamp}] ERROR in {stage} for scale {scale!r}: {exc!r}"
    print(msg)
    run_error_records.append(dict(stage=stage, scale=scale, error=repr(exc)))
    with open(cfa_dir / 'run_errors.log', 'a') as ef:
        ef.write(msg + '\n')
        ef.write(traceback.format_exc() + '\n')

## Count how many cpus I have, then decide how many I want to use; define how many iterations for CFA (decrease for debugging)

In [6]:
def get_architecture():
    # Get the raw machine architecture string
    arch = platform.machine().lower()
    
    if "arm" in arch or "aarch" in arch:
        return "ARM"
    elif "x86" in arch or "amd" in arch or "i386" in arch or "i686" in arch:
        return "x86"
    else:
        return f"Unknown ({arch})"

In [7]:
total_cpus = os.cpu_count()
# account for hyperthreading
arch = get_architecture()
if arch == 'ARM':
    cpus_to_use = total_cpus - 2
else:
    cpus_to_use = total_cpus // 2 -1
global cpus_to_use
print(f"\nGoing to use {cpus_to_use} CPUs for CFA heavy-lifting\n")

num_iter = 1000
global num_iter


Going to use 14 CPUs for CFA heavy-lifting



## SET SEEDS !!!!!!!!!!

In [8]:
#rngkind = "L'Ecuyer-CMRG"
random.seed(12345)

In [9]:
ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
ro.r('set.seed(12345)')

<rpy2.rinterface_lib.sexp.NULLType object at 0x133cb20d0> [0]

### TEST THE SEEDS!!!!!!!!

In [10]:
for i in range(5):
    print(random.random())
# after kernel restart, this should be 
# 0.41661987254534116
# 0.010169169457068361
# 0.8252065092537432
# 0.2986398551995928
# 0.3684116894884757

0.41661987254534116
0.010169169457068361
0.8252065092537432
0.2986398551995928
0.3684116894884757


In [11]:
ro.r('rnorm(5)')
# after kernel restart, this should be 
# -1.457850350316457	-0.45246126454182867	0.3650586371545244	-1.57091128601566	1.1419085835874878

-1.457850350316457,-0.45246126454182867,0.3650586371545244,-1.57091128601566,1.1419085835874878


### I'M ALSO SETTING THE SAME SEEDS EVERY TIME I RUN THE HELPED CFA FUNCTION, JUST IN CASE!!!!!

# Functions

## CFA helper functions

In [12]:
from hitop_cfa import (
    build_luts,
    check_hitop_ids,
    cfa_helper_func,
    run_specific_cfa,
    do_three_way_cfa_stepwise_mi,
    do_three_way_cfa_stepwise_scalar,
    do_stepwise_scalar_from_metric_run,
    load_metric_run,
    exhaustive_cfa_ablations,
    set_seeds,
    silence_r,
)

# item-text lookups (was load_item_lookup + inline lut construction)
_luts = build_luts(path_to_item_lookup, path_to_cogmood_questions)
item_lookup = _luts['item_lookup']
item_lut = _luts['item_lut']
phq_lut = _luts['phq_lut']
gad_lut = _luts['gad_lut']
baars_lut = _luts['baars_lut']

/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## CFA wrapper functions

# Run Main Code

## Load preprocessed data and concatinate

In [13]:
# load (validation vs enriched two-way variant)
data_val = pd.read_csv(path_save_val)
data_en = pd.read_csv(path_save_dat_en_grid1st_full)
# concat order MUST match the 3-way notebook (val first): permutation
# draws depend on row order, so this keeps the reused V_EN baseline rows
# and any fresh tests seed-identical to the 3-way pipeline
data_val_enriched = pd.concat([data_val, data_en])

# Single dataset pair; the key strings keep the 3-way labels so the
# reused baseline rows and history column names line up.
datasets_runspecific = {'V_EN': data_val_enriched}
datasets_stepwise = {'val_en': data_val_enriched}


# Invariance analyses (measEq / Wu & Estabrook 2016 ladder)

All invariance models are generated by `semTools::measEq.syntax` with `ID.cat = "Wu.Estabrook.2016"` and `ID.fac = "std.lv"` — plain `group.equal` shortcuts are vacuous at scalar/strict for ordinal indicators, and marker identification is underidentified under Wu–Estabrook (see `HANDOFF_measeq_fix.md`). The level ladder is:

**configural → thresholds → metric → scalar → strict**

where metric = thresholds + loadings, scalar = + intercepts, strict = + residuals. `assert_level_adds_df` runs before every permutation delta test, so a vacuous comparison raises instead of silently passing. There is no marker item under `std.lv`.

## Original scale formulas

In [14]:
orig_items = {
    'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop92 + hitop93 + hitop123 + hitop157 + hitop182 + hitop230 + hitop246',
    'anxious_worry': 'anxious_worry =~hitop20 + hitop34 + hitop89 + hitop203 + hitop240 + hitop248 + hitop265',
    'appetite_gain': 'appetite_gain =~hitop120 + hitop141 + hitop243 + hitop275',
    'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
    'cognitive_problems': 'cognitive_problems =~hitop67 + hitop159 + hitop189 + hitop142',
    'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
    'indecisiveness': 'indecisiveness =~hitop21 + hitop90 + hitop95',
    'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
    'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
    'separation_insecurity': 'separation_insecurity =~hitop40 + hitop50 + hitop69 + hitop81 + hitop113 + hitop136 + hitop151 + hitop197',
    'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
    'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247 + hitop278',
    'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop117 + hitop124 + hitop129 + hitop204 + hitop222 + hitop236 + hitop258',
    'well_being': 'well_being =~hitop9 + hitop23 + hitop54 + hitop106 + hitop149 + hitop200 + hitop244 + hitop245 + hitop250 + hitop281'
}
            

## Baseline: 5-level ladder over the original scales (reused from the 3-way run where possible)

The val_en baseline is **not recomputed when the 3-way pipeline already ran it**: `cfa_helper_func` reseeds (`set_seeds(12345)`) at the start of every scale × pair call, so the V_EN rows recorded in `data/cfa/orig_cfa_res.csv` are bit-identical to what this notebook would produce (same items, same concatenated data, same `num_iter`, same 14 workers). Those rows are reused directly; the ladder is run fresh only for scales missing from that record (e.g. a scale that errored in the 3-way run, or if the 3-way baseline hasn't been run at all — the notebook is self-sufficient either way). Untested levels are `'NA'`/NaN as usual.

Note the **stepwise stages below always run fresh**: the 3-way stepwise searches were steered by the validation pairs, so their removal paths and cores do not transfer to a val_en-only criterion.

In [15]:
# Reuse the 3-way run's V_EN baseline rows where available (seed-identical
# tests; see the markdown above). Only scales missing from that record get
# a fresh val_en ladder here.
threeway_csv = dat_dir / 'cfa' / 'orig_cfa_res.csv'
reused = pd.DataFrame()
if threeway_csv.exists():
    reused = pd.read_csv(threeway_csv)
    reused = reused[(reused['pair'] == 'V_EN')
                    & reused['scale'].isin(orig_items)].copy()
reused_scales = set(reused['scale']) if len(reused) else set()
scales_to_run = [s for s in orig_items if s not in reused_scales]
print(f"reused V_EN baseline rows for {len(reused_scales)} scales; "
      f"running the ladder fresh for {len(scales_to_run)}: {scales_to_run}")

with open("log/mylog_2wayCFA_en_val_origscales_seed12345.txt", "w") as f:
    with redirect_stdout(f):
        fresh = []
        for scale in scales_to_run:
            items = orig_items[scale]
            # print which scale we are processing through R - this way it doesn't get saved in the log file
            ro.globalenv['scale_to_print'] = scale
            ro.r('print(scale_to_print)')
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test (per-scale try/except: one bad scale must not kill the run)
            try:
                cfa_res = run_specific_cfa(
                    whichscale=scale,
                    item_list=items_list,
                    whichcfa='strict',
                    datasets=datasets_runspecific,
                    temp_path=path_to_helpfile,
                    num_iter=num_iter,
                    cpus_to_use=cpus_to_use,
                    return_vals=True
                )
            except Exception as exc:
                record_run_error('baseline', scale, exc)
                continue
            cfa_res = pd.DataFrame(cfa_res)
            cfa_res['scale'] = scale
            fresh.append(cfa_res)
            # persist incrementally so a later crash cannot lose completed scales
            pd.concat([reused] + fresh).to_csv(
                cfa_dir / 'orig_cfa_res_in_progress.csv', index=None)
orig_cfa_res = pd.concat([reused] + fresh, ignore_index=True)
if not len(orig_cfa_res):
    raise RuntimeError('no baseline results (nothing reusable and every '
                       'fresh scale failed); see run_errors.log')

reused V_EN baseline rows for 14 scales; running the ladder fresh for 0: []


In [16]:
# convert the p-value columns explicitly (extract_p returns floats for
# tested levels and the string 'NA' for untested ones; the old
# astype(float, errors='ignore') is a silent no-op under pandas 3)
for col in ['pconfig', 'pthresholds', 'pmetric', 'pscalar', 'pstrict']:
    orig_cfa_res[col] = pd.to_numeric(orig_cfa_res[col], errors='coerce')

In [17]:
orig_cfa_res.to_csv(cfa_dir / 'orig_cfa_res.csv', index=None)

## Derive `scales_failing_metric` from the baseline results

Replaces the old hardcoded list — the failure pattern may shift under the corrected measEq models. A scale needs the stepwise metric search unless the `val_en` pair reached metric and passed it (`pmetric` numeric and ≥ .05). Tested levels are floats (`extract_p` reads the exact permutation p from the permuteMeasEq object); levels never reached are recorded as the string `'NA'` (e.g. when configural or thresholds failed), so coerce with `pd.to_numeric(..., errors='coerce')` before filtering; a coerced NaN counts as failing.

In [18]:
# pmetric is numeric (converted above); NaN (level never reached) -> False
metric_pass_by_scale = orig_cfa_res['pmetric'].ge(0.05).groupby(orig_cfa_res['scale']).all()
failing = set(metric_pass_by_scale.index[~metric_pass_by_scale])

# scales VERIFIED metric-invariant on the full item set at baseline -- the
# scalar-continuation loop uses this to decide when the full scale is a
# valid metric core (a scale that ERRORED at baseline is in neither set
# and must not be assumed invariant)
baseline_metric_passers = set(metric_pass_by_scale.index[metric_pass_by_scale])

# keep orig_items order for reproducible loop order
scales_failing_metric = [s for s in orig_items if s in failing]
print(f"{len(scales_failing_metric)} of {len(orig_items)} scales fail val_en "
      f"metric invariance and go to the stepwise search:")
scales_failing_metric

7 of 14 scales fail val_en metric invariance and go to the stepwise search:


['anhedonic_depression',
 'anxious_worry',
 'cognitive_problems',
 'hyposomnia',
 'indecisiveness',
 'social_anxiety',
 'well_being']

## Stepwise metric search (scales failing val_en metric)

for a set of items:
    if it's val_en config, val_en thresholds, and val_en metric invariant:
        return set of items and list of removed items
    if it's not val_en config invariant:
        evaluate configural invariance on all k-item ablations (smallest k first; every item is a candidate — no marker under std.lv)
        among ablations that are val_en config invariant, pick by CFI (0.001 tolerance) → TLI (0.001 tolerance) → RMSEA
        restart the loop with the selected set of items
    if it's val_en config invariant, but not val_en thresholds:
        remove the item with the highest aggregated threshold-equality modification index
        restart the loop with the selected set of items
    if it's val_en thresholds invariant, but not val_en metric:
        remove the item with the highest aggregated loading modification index
        restart the loop with the selected set of items

In [19]:
# silence R to clean up messages
# be careful doing this, you might miss important warnings
silence_r()

In [20]:
# scales_failing_metric is derived from the baseline results above
stepwise_res = []
histories = []
for scale in scales_failing_metric:
    try:
        final, removed, history = do_three_way_cfa_stepwise_mi(
            scale,
            orig_items=orig_items,
            datasets=datasets_stepwise,
            temp_path=path_to_helpfile,
            num_iter=num_iter,
            cpus_to_use=cpus_to_use,
            min_items=3,
        )
    except Exception as exc:
        # one bad scale must not kill the run; no history pickle is written
        # for this scale, and the scalar loop below treats a metric-failing
        # scale without stepwise output as an error, not a full-scale core
        record_run_error('stepwise_metric', scale, exc)
        continue
    if final is not None:
        final_items = [item_lut[item_no] for item_no in final]
        removed_items = [item_lut[item_no] for item_no in removed]
        row = dict(
            scale=scale,
            item_nos=final,
            removed_nos=removed,
            items=final_items,
            removed=removed_items
        )
    else:
        row = dict(
            scale=scale
        )

    stepwise_res.append(row)
    pd.DataFrame(stepwise_res).to_pickle(cfa_dir / 'stepwise_in_progress.pkl')
    history['scale'] = scale
    history.to_pickle(cfa_dir / f'{scale}_history.pkl')
    histories.append(history)


STEPWISE CFA: ANHEDONIC_DEPRESSION

--- Iteration 0: 10 items ---
Current: ['hitop39', 'hitop77', 'hitop84', 'hitop92', 'hitop93', 'hitop123', 'hitop157', 'hitop182', 'hitop230', 'hitop246']
Testing current item set...


Thresholds OK but metric failed; running loading-MI-based removal...
  Aggregated loading MIs (summed over failing comparisons):
    hitop39: 11.304
    hitop157: 7.403
    hitop92: 3.831
    hitop230: 2.463
    hitop77: 2.132
    hitop123: 2.084
    hitop84: 0.600
    hitop182: 0.031
    hitop93: 0.009
    hitop246: 0.003
  -> Removing: hitop39

--- Iteration 1: 9 items ---
Current: ['hitop77', 'hitop84', 'hitop92', 'hitop93', 'hitop123', 'hitop157', 'hitop182', 'hitop230', 'hitop246']
Testing current item set...



*** 3-way metric invariance achieved with 9 items ***

STEPWISE CFA: ANXIOUS_WORRY

--- Iteration 0: 7 items ---
Current: ['hitop20', 'hitop34', 'hitop89', 'hitop203', 'hitop240', 'hitop248', 'hitop265']
Testing current item set...


Thresholds OK but metric failed; running loading-MI-based removal...
  Aggregated loading MIs (summed over failing comparisons):
    hitop34: 7.358
    hitop20: 5.922
    hitop89: 4.466
    hitop240: 1.104
    hitop203: 0.766
    hitop248: 0.605
    hitop265: 0.023
  -> Removing: hitop34

--- Iteration 1: 6 items ---
Current: ['hitop20', 'hitop89', 'hitop203', 'hitop240', 'hitop248', 'hitop265']
Testing current item set...


Thresholds OK but metric failed; running loading-MI-based removal...
  Aggregated loading MIs (summed over failing comparisons):
    hitop89: 6.420
    hitop20: 3.733
    hitop265: 0.361
    hitop203: 0.219
    hitop240: 0.192
    hitop248: 0.100
  -> Removing: hitop89

--- Iteration 2: 5 items ---
Current: ['hitop20', 'hitop203', 'hitop240', 'hitop248', 'hitop265']
Testing current item set...


...

    min_cfi=0.9224 min_tli=0.8965 max_rmsea=0.1299 all_config_passed=False
  Trying drop of hitop281 -> 9 items


    min_cfi=0.9383 min_tli=0.9177 max_rmsea=0.1138 all_config_passed=False
  Level 1: 5 of 10 combos achieve 3-way configural invariance.
  -> Removing combo ('hitop245',) (ablation_level=1, reason=cfi_max_min)

--- Iteration 1: 9 items ---
Current: ['hitop9', 'hitop23', 'hitop54', 'hitop106', 'hitop149', 'hitop200', 'hitop244', 'hitop250', 'hitop281']
Testing current item set...


Thresholds OK but metric failed; running loading-MI-based removal...
  Aggregated loading MIs (summed over failing comparisons):
    hitop54: 11.405
    hitop106: 7.487
    hitop200: 4.460
    hitop244: 4.179
    hitop250: 1.387
    hitop281: 0.918
    hitop9: 0.133
    hitop23: 0.013
    hitop149: 0.003
  -> Removing: hitop54

--- Iteration 2: 8 items ---
Current: ['hitop9', 'hitop23', 'hitop106', 'hitop149', 'hitop200', 'hitop244', 'hitop250', 'hitop281']
Testing current item set...


Thresholds OK but metric failed; running loading-MI-based removal...
  Aggregated loading MIs (summed over failing comparisons):
    hitop244: 6.663
    hitop106: 6.006
    hitop200: 3.208
    hitop281: 2.520
    hitop250: 0.555
    hitop149: 0.178
    hitop23: 0.136
    hitop9: 0.013
  -> Removing: hitop244

--- Iteration 3: 7 items ---
Current: ['hitop9', 'hitop23', 'hitop106', 'hitop149', 'hitop200', 'hitop250', 'hitop281']
Testing current item set...



*** 3-way metric invariance achieved with 7 items ***


In [21]:
stepwise_res = pd.DataFrame(stepwise_res)

In [22]:
stepwise_res

,scale,item_nos,removed_nos,items,removed
0,anhedonic_depression,"[hitop77, hitop84, hitop92, hitop93, hitop123, hitop157, hitop182, hitop230, hitop246]",[hitop39],"[I didn’t look forward to seeing friends or family., I felt depressed., It took a lot of effort ...",[It felt like there wasn’t anything interesting or fun to do.]
1,anxious_worry,"[hitop20, hitop203, hitop240, hitop248, hitop265]","[hitop34, hitop89]","[I felt tense., I felt very stressed., I felt nervous and ""on edge""., I worried about almost eve...","[Thoughts were racing through my head., I had a lot of nervous energy.]"
2,cognitive_problems,NaN,NaN,NaN,NaN
3,hyposomnia,"[hitop99, hitop181, hitop66, hitop231]",[hitop5],"[I needed much less sleep than usual., I felt like I could keep going and going without ever get...",[I had days when I never got tired.]
4,indecisiveness,NaN,NaN,NaN,NaN
5,social_anxiety,"[hitop114, hitop117, hitop124, hitop129, hitop204, hitop222, hitop236, hitop258]","[hitop17, hitop1]","[I had difficulty making eye contact with others., I felt socially awkward., I avoided situation...","[I was uncomfortable meeting new people., I felt shy around other people.]"
6,well_being,"[hitop9, hitop23, hitop106, hitop149, hitop200, hitop250, hitop281]","[hitop245, hitop54, hitop244]","[I felt like I was having a lot of fun., I felt cheerful., I was proud of myself., I felt good a...","[I looked forward to things with enjoyment., It was easy for me to laugh., I felt optimistic.]"


In [23]:
stepwise_res.to_pickle(cfa_dir / 'stepwise.pkl')

## Scalar continuation from the metric cores

The invariance target is **scalar** (thresholds + loadings + intercepts): the between-sample latent mean comparisons are only valid under scalar invariance. For every scale we continue from its metric core toward a scalar core; if the continuation bottoms out, the **metric core is that scale's deliverable** (reporting rule: ICCs may use metric-fallback cores, mean comparisons are reported only for scales with scalar cores).

Mechanics under Wu–Estabrook: the scalar delta test is the **param-free omnibus permutation** (W&E fixes group-2 intercepts back to 0 rather than equating them, so `param="intercepts"` has no constraints to point at), and item removal at the scalar level is driven by `lavaan::modindices()` intercept MIs on the scalar fit (score test for freeing each fixed group-2 intercept).

Per scale: `load_metric_run` reads the metric core from this run's pickles (`{scale}_history.pkl`, falling back to `stepwise.pkl` — measEq-era pickles only). Scales with no stepwise metric run (`FileNotFoundError`) passed metric on the full item set at baseline, so the full scale is their metric core. Lower-level permutation tests were already run on exactly these items/data/seeds, so the first iteration assumes them and runs only the scalar test (`assume_metric_invariant=True`, the default); the df ladder is still asserted at every level.

The resulting `stepwise_scalar.pkl` is the **final-cores table** consumed by NB_3_ICC: one row per scale with `core_level` (`'scalar'`, `'metric'`, or `None`), the core item set, and the items removed relative to the original scale.

In [24]:
scalar_stepwise_res = []
scalar_histories = []
for scale in orig_items:
    items_list = orig_items[scale].split("=~", 1)[1].strip().split(" + ")
    try:
        # metric core from this run's measEq-era stepwise pickles
        metric_core, _metric_history = load_metric_run(cfa_dir, scale)
    except FileNotFoundError:
        if scale in baseline_metric_passers:
            # no stepwise metric run exists because the scale passed metric
            # on the full item set at baseline; the full scale is its core
            metric_core = items_list
        else:
            # no stepwise output AND no verified baseline pass: the scale
            # errored upstream -- do NOT assume the full scale is invariant
            record_run_error(
                'scalar_continuation', scale,
                RuntimeError('no metric-stepwise output and no verified '
                             'baseline metric pass; upstream stage errored'))
            history = pd.DataFrame([{
                'iteration': 0, 'phase': 'final', 'n_items': 0,
                'items': tuple(), 'action': 'upstream_error',
            }])
            row = dict(scale=scale, core_level=None, error='upstream_error')
            scalar_stepwise_res.append(row)
            pd.DataFrame(scalar_stepwise_res).to_pickle(
                cfa_dir / 'stepwise_scalar_in_progress.pkl')
            history['scale'] = scale
            history.to_pickle(cfa_dir / f'{scale}_scalar_history.pkl')
            scalar_histories.append(history)
            continue

    if metric_core is None:
        # the stepwise metric search found no invariant core: no deliverable
        print(f"[{scale}] metric search found no invariant core; "
              f"no scalar continuation possible")
        history = pd.DataFrame([{
            'iteration': 0, 'phase': 'final', 'n_items': 0,
            'items': tuple(), 'action': 'no_metric_core',
        }])
        row = dict(scale=scale, core_level=None)
    else:
        try:
            final, removed, history = do_three_way_cfa_stepwise_scalar(
                scale,
                metric_core,
                datasets=datasets_stepwise,
                temp_path=path_to_helpfile,
                num_iter=num_iter,
                cpus_to_use=cpus_to_use,
                min_items=3,
            )
            error = None
        except Exception as exc:
            # the metric core is still a verified deliverable; fall back to
            # it, flag the error, and keep the run alive
            record_run_error('scalar_continuation', scale, exc)
            final, removed = None, []
            error = 'scalar_search_error'
            history = pd.DataFrame([{
                'iteration': 0, 'phase': 'final',
                'n_items': len(metric_core), 'items': tuple(metric_core),
                'action': 'scalar_search_error',
            }])
        if final is not None:
            core, core_level = final, 'scalar'
        else:
            # scalar continuation bottomed out (or errored): the metric
            # core is the scale's deliverable (metric fallback)
            core, core_level = metric_core, 'metric'
        removed_from_orig = [ii for ii in items_list if ii not in core]
        row = dict(
            scale=scale,
            core_level=core_level,
            item_nos=core,
            removed_nos=removed_from_orig,
            items=[item_lut[item_no] for item_no in core],
            removed=[item_lut[item_no] for item_no in removed_from_orig],
            metric_item_nos=metric_core,
            scalar_removed_nos=removed,  # removed during the scalar stage only
            error=error,
        )

    scalar_stepwise_res.append(row)
    pd.DataFrame(scalar_stepwise_res).to_pickle(
        cfa_dir / 'stepwise_scalar_in_progress.pkl')
    history['scale'] = scale
    history.to_pickle(cfa_dir / f'{scale}_scalar_history.pkl')
    scalar_histories.append(history)


STEPWISE SCALAR: ANHEDONIC_DEPRESSION

--- Iteration 0: 9 items ---
Current: ['hitop77', 'hitop84', 'hitop92', 'hitop93', 'hitop123', 'hitop157', 'hitop182', 'hitop230', 'hitop246']
Testing current item set (configural + thresholds + metric assumed from completed metric run; scalar test only)...


Scalar failed; running intercept-MI-based removal...
  Aggregated intercept MIs (summed over failing comparisons):
    hitop93: 23.795
    hitop157: 13.328
    hitop123: 11.674
    hitop246: 8.273
    hitop92: 7.682
    hitop77: 5.496
    hitop182: 5.441
    hitop84: 1.240
    hitop230: 0.139
  -> Removing: hitop93

--- Iteration 1: 8 items ---
Current: ['hitop77', 'hitop84', 'hitop92', 'hitop123', 'hitop157', 'hitop182', 'hitop230', 'hitop246']
Testing current item set up to scalar...


Scalar failed; running intercept-MI-based removal...
  Aggregated intercept MIs (summed over failing comparisons):
    hitop246: 12.038
    hitop182: 9.634
    hitop157: 8.318
    hitop123: 7.676
    hitop84: 3.848
    hitop92: 3.507
    hitop77: 3.138
    hitop230: 0.177
  -> Removing: hitop246

--- Iteration 2: 7 items ---
Current: ['hitop77', 'hitop84', 'hitop92', 'hitop123', 'hitop157', 'hitop182', 'hitop230']
Testing current item set up to scalar...


Scalar failed; running intercept-MI-based removal...
  Aggregated intercept MIs (summed over failing comparisons):
    hitop182: 12.503
    hitop123: 6.405
    hitop157: 6.064
    hitop84: 5.658
    hitop77: 1.906
    hitop92: 1.695
    hitop230: 0.524
  -> Removing: hitop182

--- Iteration 3: 6 items ---
Current: ['hitop77', 'hitop84', 'hitop92', 'hitop123', 'hitop157', 'hitop230']
Testing current item set up to scalar...


Scalar failed; running intercept-MI-based removal...
  Aggregated intercept MIs (summed over failing comparisons):
    hitop84: 8.417
    hitop123: 4.049
    hitop157: 2.737
    hitop230: 1.737
    hitop77: 0.507
    hitop92: 0.142
  -> Removing: hitop84

--- Iteration 4: 5 items ---
Current: ['hitop77', 'hitop92', 'hitop123', 'hitop157', 'hitop230']
Testing current item set up to scalar...


...


*** 3-way scalar invariance achieved with 5 items ***

STEPWISE SCALAR: WELL_BEING

--- Iteration 0: 7 items ---
Current: ['hitop9', 'hitop23', 'hitop106', 'hitop149', 'hitop200', 'hitop250', 'hitop281']
Testing current item set (configural + thresholds + metric assumed from completed metric run; scalar test only)...


Scalar failed; running intercept-MI-based removal...
  Aggregated intercept MIs (summed over failing comparisons):
    hitop281: 20.033
    hitop9: 18.282
    hitop23: 12.735
    hitop250: 8.064
    hitop106: 1.214
    hitop200: 0.469
    hitop149: 0.160
  -> Removing: hitop281

--- Iteration 1: 6 items ---
Current: ['hitop9', 'hitop23', 'hitop106', 'hitop149', 'hitop200', 'hitop250']
Testing current item set up to scalar...


Scalar failed; running intercept-MI-based removal...
  Aggregated intercept MIs (summed over failing comparisons):
    hitop9: 12.891
    hitop250: 10.449
    hitop23: 7.688
    hitop106: 4.011
    hitop149: 1.955
    hitop200: 0.007
  -> Removing: hitop9

--- Iteration 2: 5 items ---
Current: ['hitop23', 'hitop106', 'hitop149', 'hitop200', 'hitop250']
Testing current item set up to scalar...


Scalar failed; running intercept-MI-based removal...
  Aggregated intercept MIs (summed over failing comparisons):
    hitop23: 13.983
    hitop250: 4.973
    hitop106: 2.350
    hitop149: 0.364
    hitop200: 0.302
  -> Removing: hitop23

--- Iteration 3: 4 items ---
Current: ['hitop106', 'hitop149', 'hitop200', 'hitop250']
Testing current item set up to scalar...



*** 3-way scalar invariance achieved with 4 items ***


In [25]:
scalar_stepwise_res = pd.DataFrame(scalar_stepwise_res)
scalar_stepwise_res

,scale,core_level,item_nos,removed_nos,items,removed,metric_item_nos,scalar_removed_nos,error
0,anhedonic_depression,scalar,"[hitop77, hitop123, hitop230]","[hitop39, hitop84, hitop92, hitop93, hitop157, hitop182, hitop246]","[I didn’t look forward to seeing friends or family., Nothing made me laugh., I felt emotionally ...","[It felt like there wasn’t anything interesting or fun to do., I felt depressed., It took a lot ...","[hitop77, hitop84, hitop92, hitop93, hitop123, hitop157, hitop182, hitop230, hitop246]","[hitop93, hitop246, hitop182, hitop84, hitop92, hitop157]",NaN
1,anxious_worry,scalar,"[hitop20, hitop203, hitop265]","[hitop34, hitop89, hitop240, hitop248]","[I felt tense., I felt very stressed., I was overwhelmed by anxiety.]","[Thoughts were racing through my head., I had a lot of nervous energy., I felt nervous and ""on e...","[hitop20, hitop203, hitop240, hitop248, hitop265]","[hitop248, hitop240]",NaN
2,appetite_gain,metric,"[hitop120, hitop141, hitop243, hitop275]",[],"[I could not keep myself from eating., I thought a lot about food., I stuffed myself with food.,...",[],"[hitop120, hitop141, hitop243, hitop275]","[hitop141, hitop275]",NaN
3,appetite_loss,metric,"[hitop280, hitop283, hitop109]",[],"[My appetite was poor., I lost a significant amount of weight without even trying., I did not fe...",[],"[hitop280, hitop283, hitop109]",[hitop280],NaN
4,cognitive_problems,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,hyposomnia,scalar,"[hitop99, hitop181, hitop66]","[hitop5, hitop231]","[I needed much less sleep than usual., I felt like I could keep going and going without ever get...","[I had days when I never got tired., I felt like I could go for days without sleeping.]","[hitop99, hitop181, hitop66, hitop231]",[hitop231],NaN
6,indecisiveness,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,insomnia,scalar,"[hitop160, hitop254, hitop268]",[hitop261],"[I had trouble staying asleep., I slept very poorly., I lay awake for a long time before falling...",[I woke up early and could not get back to sleep.],"[hitop160, hitop254, hitop261, hitop268]",[hitop261],NaN
8,panic,scalar,"[hitop15, hitop104, hitop126, hitop211, hitop215]",[hitop257],"[I was short of breath., I felt nauseated., My heart was racing or pounding., I was trembling or...",[I felt dizzy or lightheaded.],"[hitop15, hitop104, hitop126, hitop211, hitop215, hitop257]",[hitop257],NaN
9,separation_insecurity,scalar,"[hitop40, hitop50, hitop69, hitop81, hitop136, hitop151, hitop197]",[hitop113],"[I felt insecure about important relationships in my life., I wanted someone else to make decisi...",[I could not handle rejection.],"[hitop40, hitop50, hitop69, hitop81, hitop113, hitop136, hitop151, hitop197]",[hitop113],NaN


In [26]:
scalar_stepwise_res.to_pickle(cfa_dir / 'stepwise_scalar.pkl')

## Strict invariance report on the final cores

Report-only (per the handoff's reporting rule): each scale's final core (scalar core, or metric fallback) is run up the full ladder to **strict** on the `val_en` pair. Strict pass/fail is reported in the manuscript but drives no item removal. The strict delta test is the param-free omnibus permutation (same W&E fixing logic as scalar).

In [27]:
with open(log_dir / "mylog_2wayCFA_en_val_finalcores_strict_seed12345.txt", "w") as f:
    with redirect_stdout(f):
        strict_report = []
        for row in scalar_stepwise_res.itertuples():
            if row.core_level not in ('scalar', 'metric'):
                continue
            try:
                res = run_specific_cfa(
                    whichscale=row.scale,
                    item_list=list(row.item_nos),
                    whichcfa='strict',
                    datasets=datasets_runspecific,
                    temp_path=path_to_helpfile,
                    num_iter=num_iter,
                    cpus_to_use=cpus_to_use,
                    return_vals=True,
                )
            except Exception as exc:
                record_run_error('strict_report', row.scale, exc)
                continue
            res = pd.DataFrame(res)
            res['scale'] = row.scale
            res['core_level'] = row.core_level
            strict_report.append(res)
            # persist incrementally
            pd.concat(strict_report).replace("NA", pd.NA).to_csv(
                cfa_dir / 'final_cores_strict_report_in_progress.csv',
                index=None)
strict_report = pd.concat(strict_report) if strict_report else pd.DataFrame()
strict_report = strict_report.replace("NA", pd.NA)
strict_report.to_csv(cfa_dir / 'final_cores_strict_report.csv', index=None)
strict_report

,pair,pconfig,pthresholds,pmetric,pscalar,pstrict,scale,core_level
0,V_EN,0.688,0.697,0.456,0.382,0.731,anhedonic_depression,scalar
0,V_EN,0.017,0.381,0.698,0.765,0.711,anxious_worry,scalar
0,V_EN,0.871,0.409,0.123,0.000,<NA>,appetite_gain,metric
0,V_EN,0.324,0.432,0.074,0.004,<NA>,appetite_loss,metric
0,V_EN,0.877,0.141,0.264,0.604,0.036,hyposomnia,scalar
0,V_EN,0.875,0.615,0.910,0.361,0.331,insomnia,scalar
0,V_EN,0.077,0.689,0.519,0.449,0.064,panic,scalar
0,V_EN,0.111,0.898,0.136,0.104,0.0,separation_insecurity,scalar
0,V_EN,0.993,0.296,0.574,0.217,0.02,shame_guilt,scalar
0,V_EN,0.150,0.946,0.389,0.427,0.001,situational_phobia,scalar


In [28]:
# ---- Overnight run summary ----
print(f"scales in baseline results:      "
      f"{orig_cfa_res['scale'].nunique()} / {len(orig_items)}")
print(f"scales failing val_en metric:     {len(scales_failing_metric)}")
n_scalar = int((scalar_stepwise_res.core_level == 'scalar').sum())
n_metric = int((scalar_stepwise_res.core_level == 'metric').sum())
n_none = int(scalar_stepwise_res.core_level.isnull().sum())
print(f"final cores: {n_scalar} scalar, {n_metric} metric fallback, "
      f"{n_none} without an invariant core")
if run_error_records:
    print(f"\n!!! {len(run_error_records)} ERROR(S) recorded during this run "
          f"(details + tracebacks in {cfa_dir / 'run_errors.log'}):")
    for rec in run_error_records:
        print(f"  [{rec['stage']}] {rec['scale']}: {rec['error']}")
else:
    print("\nno errors recorded during this run")

scales in baseline results:      14 / 14
scales failing val_en metric:     7
final cores: 10 scalar, 2 metric fallback, 2 without an invariant core

no errors recorded during this run
